In [ ]:
!pip install chromadb langchain langchain-community sentence-transformers langchain-huggingface kuzu -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 

In [ ]:
import os
from google.colab import userdata

# Retrieve the secret from Colab
hf_token = userdata.get('HUGGINGFACEHUB_API_TOKEN')

# Set it as an environment variable for LangChain
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

print("Secret loaded successfully!")

Secret loaded successfully!


In [ ]:
import re
import json
import os
import shutil
import uuid
from pathlib import Path

import kuzu
from sentence_transformers import SentenceTransformer

# --- Configuration (mirrors BE_G_Indexing) ---
JSON_PATH        = "./complete_dataset.json"  # output from the preprocessing colab
CHUNK_SIZE       = 400   # characters per chunk (~80-100 tokens)
CHUNK_STRIDE     = 50    # overlap between chunks
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
EMBED_DIM        = 384
KUZU_DB_PATH     = Path(JSON_PATH).stem + "_rag.kuzu"

# Initialize Embedding Model
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

# Initialize KuzuDB
if Path(KUZU_DB_PATH).exists():
    shutil.rmtree(KUZU_DB_PATH)

db   = kuzu.Database(KUZU_DB_PATH)
conn = kuzu.Connection(db)

# Schema: one Chunk node per text chunk, with its embedding and provenance
conn.execute(f"""
    CREATE NODE TABLE IF NOT EXISTS Chunk(
        chunk_id    STRING PRIMARY KEY,
        text        STRING,
        source_file STRING,
        page_number INT64,
        embedding   FLOAT[{EMBED_DIM}]
    )
""")

print(f"KuzuDB initialised at: {KUZU_DB_PATH}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

KuzuDB initialised at: complete_dataset_rag.kuzu


In [ ]:
def load_json(path: str) -> list[dict]:
    """
    Load the cleaned dataset JSON.
    Each record: {source_file, start_page, end_page, text: [{text, page_number}, ...]}
    Returns a flat list of page-level dicts:
        {source_file: str, page_number: int, text: str}
    """
    with open(path, encoding="utf-8") as f:
        records = json.load(f)
    pages = []
    for record in records:
        source_file = record["source_file"]
        for page_obj in record["text"]:
            pages.append({
                "source_file": source_file,
                "page_number":  page_obj["page_number"],
                "text":         page_obj["text"],
            })
    return pages


pages = load_json(JSON_PATH)
print(f"Loaded {len(pages)} pages from {JSON_PATH}")
print("Preview (first page):")
print(f"  source_file : {pages[0]['source_file']}")
print(f"  page_number : {pages[0]['page_number']}")
print(f"  text[:200]  : {pages[0]['text'][:200]}")


Loaded 3361 pages from ./complete_dataset.json
Preview (first page):
  source_file : dbms.pdf
  page_number : 32
  text[:200]  : This page intentionally left blank 


In [ ]:
def chunk_text(text: str, size: int = CHUNK_SIZE, stride: int = CHUNK_STRIDE) -> list[str]:
    """Sliding-window, sentence-aware chunker (identical to BE_G_Indexing)."""
    sentences = re.split(r"(?<=[.!?]) +", text)
    chunks = []
    current = ""

    for sent in sentences:
        if len(current) + len(sent) + 1 <= size:
            current = (current + " " + sent).strip()
            continue

        if current:
            chunks.append(current)

        if len(sent) > size:
            # Hard-split a sentence that's longer than one chunk
            for i in range(0, len(sent), size - stride):
                chunks.append(sent[i:i + size])
            current = ""
        else:
            current = sent

    if current:
        chunks.append(current)
    return chunks


In [ ]:
def ingest_pages(pages: list[dict]) -> None:
    """
    Chunk every page using chunk_text (same as BE_G_Indexing) and
    insert the resulting chunks into KuzuDB with their embeddings
    and page provenance (source_file, page_number).
    """
    BATCH_SIZE = 1000
    all_chunks = []

    for page in pages:
        for chunk in chunk_text(page["text"]):
            all_chunks.append({
                "chunk_id":    str(uuid.uuid4()),
                "text":        chunk,
                "source_file": page["source_file"],
                "page_number": page["page_number"],
            })

    total = len(all_chunks)
    print(f"Total chunks to ingest: {total}")

    print("Computing embeddings...")
    texts = [c["text"] for c in all_chunks]
    embeddings_matrix = embed_model.encode(texts, batch_size=64, show_progress_bar=True)

    INSERT_CHUNK = """
        CREATE (:Chunk {
            chunk_id:    $chunk_id,
            text:        $text,
            source_file: $source_file,
            page_number: $page_number,
            embedding:   $embedding
        })
    """

    for start in range(0, total, BATCH_SIZE):
        end = min(start + BATCH_SIZE, total)
        print(f"Inserting chunks {start + 1}-{end} of {total}...")
        for i in range(start, end):
            c = all_chunks[i]
            conn.execute(INSERT_CHUNK, parameters={
                "chunk_id":    c["chunk_id"],
                "text":        c["text"],
                "source_file": c["source_file"],
                "page_number": c["page_number"],
                "embedding":   embeddings_matrix[i].tolist(),
            })

    print(f"Successfully inserted {total} chunks.")


ingest_pages(pages)

Total chunks to ingest: 26649
Computing embeddings...


Batches:   0%|          | 0/417 [00:00<?, ?it/s]

Inserting chunks 1-1000 of 26649...
Inserting chunks 1001-2000 of 26649...
Inserting chunks 2001-3000 of 26649...
Inserting chunks 3001-4000 of 26649...
Inserting chunks 4001-5000 of 26649...
Inserting chunks 5001-6000 of 26649...
Inserting chunks 6001-7000 of 26649...
Inserting chunks 7001-8000 of 26649...
Inserting chunks 8001-9000 of 26649...
Inserting chunks 9001-10000 of 26649...
Inserting chunks 10001-11000 of 26649...
Inserting chunks 11001-12000 of 26649...
Inserting chunks 12001-13000 of 26649...
Inserting chunks 13001-14000 of 26649...
Inserting chunks 14001-15000 of 26649...
Inserting chunks 15001-16000 of 26649...
Inserting chunks 16001-17000 of 26649...
Inserting chunks 17001-18000 of 26649...
Inserting chunks 18001-19000 of 26649...
Inserting chunks 19001-20000 of 26649...
Inserting chunks 20001-21000 of 26649...
Inserting chunks 21001-22000 of 26649...
Inserting chunks 22001-23000 of 26649...
Inserting chunks 23001-24000 of 26649...
Inserting chunks 24001-25000 of 26649.

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
llm_endpoint = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation",
    max_new_tokens=2048,
)
llm = ChatHuggingFace(llm=llm_endpoint)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import numpy as np

def query_rag(question: str, k: int = 10):
    """
    Embeds the question, retrieves the top-k most similar chunks from KuzuDB
    by cosine similarity, and generates an answer using the LLM.
    """

    # 1. Embedding & Retrieval
    def embed_and_retrieve(q: str):
        # Step A: Embed the question
        query_vector = embed_model.encode(q).tolist()

        # Step B: Fetch all chunks with their embeddings from KuzuDB
        result = conn.execute("""
            MATCH (c:Chunk)
            RETURN c.chunk_id, c.text, c.embedding
        """)

        # Step C: Compute cosine similarity and keep top-k
        rows = []
        while result.has_next():
            row = result.get_next()
            rows.append((row[1], np.array(row[2])))  # (text, embedding)

        q_vec = np.array(query_vector)
        scored = [
            (text, float(np.dot(q_vec, emb) / (np.linalg.norm(q_vec) * np.linalg.norm(emb) + 1e-9)))
            for text, emb in rows
        ]
        top_k = sorted(scored, key=lambda x: x[1], reverse=True)[:k]

        # Step D: Format retrieved chunks into a single context string
        return "\n\n".join(text for text, _ in top_k)

    # 2. Prompt Template
    template = """Answer the question based on the following context:
{context}

Question: {question}
"""
    prompt = ChatPromptTemplate.from_template(template)

    # 3. Build the LCEL Chain
    rag_chain = (
        {"context": embed_and_retrieve, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # 4. Execute
    return rag_chain.invoke(question)

In [ ]:
user_question = "How do functional dependencies lead to normalization, and how does the resulting schema affect index design?"

# Call the function
answer = query_rag(user_question)

# Print the returned string
print("--- LLM ANSWER ---")
print(answer)

--- LLM ANSWER ---
**1.  Functional dependencies → normalization**

| Step | What the FD tells us | How it drives the decomposition |
|------|----------------------|---------------------------------|
| **Identify keys** | A set of attributes X that functionally determines all others (X → all attributes) is a *candidate key*. | The primary key of each relation is chosen from the candidate keys. |
| **Detect anomalies** | If a non‑key attribute Y is partially or transitively dependent on a key, the relation violates 2NF/3NF/BCNF. | The violation is a signal that the relation must be split. |
| **Apply normal‑form tests** | 1NF: atomic values.<br>2NF: no partial dependence of a non‑key on part of a composite key.<br>3NF: no transitive dependence of a non‑key on the key.<br>BCNF: every determinant is a superkey. | For each violation we perform a *loss‑less join* decomposition that removes the offending dependency. The classic algorithms (e.g., the BCNF decomposition algorithm) are guided e